Import Libraries

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, sha2, concat_ws,
    row_number, coalesce, expr
)
from datetime import datetime
from pyspark.sql.window import Window
from delta.tables import DeltaTable


print("✓ Spark session ready — Delta Lake, auto-merge, CDF enabled")

Products Table

In [0]:
# ── Schema with SCD Type 2 audit columns ──
product_schema = StructType([
    StructField("product_id",     IntegerType(),  False),
    StructField("sku",            StringType(),   False),
    StructField("name",           StringType(),   False),
    StructField("category",       StringType(),   True),
    StructField("price",          DoubleType(),   True),
    StructField("stock_qty",      IntegerType(),  True),
    StructField("warehouse",      StringType(),   True),
    StructField("is_current",     BooleanType(),  True),
    StructField("effective_date", TimestampType(), True),
    StructField("end_date",       TimestampType(), True),
    StructField("row_hash",       StringType(),   True),
])

products = [
    (101, "SKU-A1", "Wireless Mouse", "Electronics", 29.99, 500, "WH-WEST",
     True, datetime(2024, 1, 1, 0, 0, 0), None, "aabbcc01"),

    (102, "SKU-B2", "Mechanical Keyboard", "Electronics", 89.99, 300, "WH-WEST",
     True, datetime(2024, 1, 1, 0, 0, 0), None, "aabbcc02"),

    (103, "SKU-C3", "USB-C Hub", "Accessories", 49.99, 1200, "WH-EAST",
     True, datetime(2024, 1, 1, 0, 0, 0), None, "aabbcc03"),

    (104, "SKU-D4", "Webcam 1080p", "Electronics", 59.99, 800, "WH-EAST",
     True, datetime(2024, 1, 1, 0, 0, 0), None, "aabbcc04"),

    (105, "SKU-E5", "Monitor Stand", "Furniture", 39.99, 200, "WH-NORTH",
     True, datetime(2024, 1, 1, 0, 0, 0), None, "aabbcc05"),
]

products_df = spark.createDataFrame(products, product_schema)

products_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("category") \
    .option("overwriteSchema", "true") \
    .saveAsTable("default.products")

print("✓ products table created — partitioned by category")
spark.table("default.products") \
    .orderBy("product_id").show(truncate=False)

In [0]:
spark.sql("""
ALTER TABLE default.products
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

Products_Update Table with CDC

In [0]:
# ── Incoming CDC records ──
# Note: "supplier" is a NEW column → triggers schema evolution
update_schema = StructType([
    StructField("product_id", IntegerType(),  False),
    StructField("sku",        StringType(),   False),
    StructField("name",       StringType(),   False),
    StructField("category",   StringType(),   True),
    StructField("price",      DoubleType(),   True),
    StructField("stock_qty",  IntegerType(),  True),
    StructField("warehouse",  StringType(),   True),
    StructField("supplier",   StringType(),   True),   # ← NEW COLUMN
    StructField("cdc_op",     StringType(),   False),   # U/I/D
])

updates = [
    # UPDATE — price change on Mouse
    (101, "SKU-A1", "Wireless Mouse",     "Electronics", 24.99, 450,
     "WH-WEST", "LogiTech",  "U"),
    # UPDATE — moved to different warehouse
    (103, "SKU-C3", "USB-C Hub",          "Accessories", 49.99, 1100,
     "WH-SOUTH", "Anker",    "U"),
    # SOFT DELETE — webcam discontinued
    (104, "SKU-D4", "Webcam 1080p",       "Electronics", 59.99, 0,
     "WH-EAST", None,        "D"),
    # INSERT — brand new product
    (106, "SKU-F6", "Ergonomic Chair",    "Furniture",   299.99, 50,
     "WH-NORTH", "HermanMiller", "I"),
    # INSERT — another new product
    (107, "SKU-G7", "Thunderbolt Dock",   "Accessories", 179.99, 150,
     "WH-EAST", "CalDigit",  "I"),
    # DUPLICATE UPDATE — test dedup (same product_id)
    (101, "SKU-A1", "Wireless Mouse Pro", "Electronics", 34.99, 600,
     "WH-WEST", "LogiTech",  "U"),
]

updates_df = spark.createDataFrame(updates, update_schema)

updates_df.display()


In [0]:
window = Window.partitionBy("product_id") \
    .orderBy(col("price").desc()) 

deduped_df = updates_df \
    .withColumn("rn", row_number().over(window)).filter(col("rn") == 1) \
    .drop("rn")

deduped_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("default.product_updates")

print("product_updates ")
deduped_df.show(truncate = False)

Row Hash for Change Detection

In [0]:
# ── Build row hash on business-critical columns ──
hash_cols = ["name", "category", "price", "stock_qty", "warehouse"]

source = spark.table("default.product_updates") \
  .withColumn(
    "row_hash",
    sha2(concat_ws(
       "||" , *[
         coalesce(col(c).cast("string"), lit("NULL"))
         for c in hash_cols
        ]
    ), 256)
  ) \
  .withColumn("effective_date", current_timestamp()) \
  .withColumn("end_date", lit(None).cast("timestamp")) \
  .withColumn("is_current", lit(True))

source.select("product_id", "name", "row_hash", "cdc_op").show(truncate = 20)

In [0]:
%sql
select * from products;

STEP A — Expire old versions of changed rows
          (SCD Type 2: close the current record)

STEP B — Insert new current versions + new products
         Filter to only U (changed) and I (new)

In [0]:
target = DeltaTable.forName(spark, "default.products")

target.alias("t").merge(
    source.alias("s"),
    "t.product_id = s.product_id AND t.is_current = true"
).whenMatchedUpdate(
    # Only fire when the row hash differs 
    condition = "s.cdc_op = 'U' AND t.row_hash != s.row_hash",
    set = {
        "is_current" : lit(False),
        "end_date" : current_timestamp(),
    }
).whenMatchedUpdate(
    # Soft-delete: mark discontinued
    condition = "s.cdc_op = 'D'",
    set = {
        "is_current" : lit(False),
        "end_date" : current_timestamp()
    }
).execute()

print("✓ Step A — expired old records")

# ═══════════════════════════════════════════════════
#  STEP B — Insert new current versions + new products
#           Filter to only U (changed) and I (new)
# ═══════════════════════════════════════════════════
new_rows = source.filter(col("cdc_op").isin("U", "I")) \
    .drop("cdc_op") \
    .withColumn("is_current", lit(True)) \
    .withColumn("effective_date", current_timestamp()) \
    .withColumn("end_date", lit(None).cast("timestamp"))

# Schema evolution: "supplier" column auto-merges
new_rows.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("default.products")

print("✓ Step B — inserted new current records")
print("\n── Final products table (all versions) ──")
spark.table("default.products") \
    .orderBy("product_id", "effective_date") \
    .show(truncate=False)